# MiniMax H3 — ComfyUI + Cloudflare Preflight First

This is the main H3 notebook. The important change is the order:

1. Install ComfyUI + required custom nodes only.
2. Start ComfyUI and the named Cloudflare Tunnel.
3. Open `https://comfy.zetbros.com` and verify the full ComfyUI UI works.
4. **STOP here by default. No H3 model weights download yet.**
5. Only after you confirm the UI works, enable the model-download gate and continue.
6. Download the H3 model stack.
7. Restart only ComfyUI once; Cloudflare stays connected.

This avoids spending Colab units on large model downloads before the public ComfyUI connection is proven.

**Important:** keep only one Colab runtime connected to this Cloudflare tunnel. Disconnect any old tunnel-test runtime first.

## 0. Mount Drive + persistence

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PERSIST_MODELS_TO_DRIVE = False   # False = faster model loading from Colab VM disk
PERSIST_OUTPUT_TO_DRIVE = True    # Keep ComfyUI output/user data across restarts
DRIVE_ROOT = '/content/drive/MyDrive/MiniMax_H3_ComfyUI'


## 1. Check GPU + safe runtime settings

In [ ]:
import subprocess, os

def sh(cmd):
    return subprocess.check_output(cmd, shell=True, text=True).strip()

gpu_name = sh('nvidia-smi --query-gpu=name --format=csv,noheader | head -n1')
vram_mb = int(sh('nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -n1'))
vram_gb = vram_mb / 1024
name = gpu_name.lower()

if 'rtx pro 6000' in name or 'blackwell' in name:
    H3_RESERVE_VRAM_GB = '6'
elif 'a100' in name:
    H3_RESERVE_VRAM_GB = '4'
elif 'l4' in name:
    H3_RESERVE_VRAM_GB = '2'
else:
    H3_RESERVE_VRAM_GB = '1'

print(f'GPU: {gpu_name} ({vram_gb:.1f} GB)')
print('Reserved VRAM:', H3_RESERVE_VRAM_GB, 'GB')


## 2. Clone/update the H3 branch

In [ ]:
%cd /content
!rm -rf /content/All-testing /content/minimax_h3_comfy
!git clone --depth 1 --branch minimax-h3-colab https://github.com/Logan17de/All-testing.git /content/All-testing
!cp -r /content/All-testing/video/minimax_h3_comfy /content/minimax_h3_comfy
%cd /content/minimax_h3_comfy


## 3. Install/update ComfyUI + H3 custom nodes — NO model weights yet

This installs ComfyUI and the custom-node code needed by your H3/Director workflows. It does **not** download the large H3 model weights.

In [ ]:
import os, subprocess, sys

os.environ['COMFY_ROOT'] = '/content/ComfyUI'
os.environ['H3_DRIVE_ROOT'] = DRIVE_ROOT
os.environ['H3_PERSIST_MODELS'] = '1' if PERSIST_MODELS_TO_DRIVE else '0'
os.environ['H3_PERSIST_OUTPUT'] = '1' if PERSIST_OUTPUT_TO_DRIVE else '0'
os.environ['H3_VRAM_MODE'] = 'auto'
os.environ['H3_RESERVE_VRAM_GB'] = H3_RESERVE_VRAM_GB
os.environ['H3_PREVIEW_METHOD'] = 'none'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!bash install_comfy_h3.sh

CUSTOM='/content/ComfyUI/custom_nodes'
def clone_or_pull(url, folder):
    path=f'{CUSTOM}/{folder}'
    if os.path.isdir(path+'/.git'):
        subprocess.run(['git','-C',path,'pull','--ff-only'], check=False)
    else:
        subprocess.run(['git','clone','--depth','1',url,path], check=True)
    req=os.path.join(path,'requirements.txt')
    if os.path.exists(req):
        subprocess.run([sys.executable,'-m','pip','install','-r',req], check=False)

clone_or_pull('https://github.com/AIMixer/ComfyUI_MiniMaxH3_Director.git','ComfyUI_MiniMaxH3_Director')
clone_or_pull('https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git','ComfyUI-VideoHelperSuite')
clone_or_pull('https://github.com/kijai/ComfyUI-KJNodes.git','ComfyUI-KJNodes')
clone_or_pull('https://github.com/pixaroma/ComfyUI-Pixaroma.git','ComfyUI-Pixaroma')
subprocess.run([sys.executable,'-m','pip','install','-U','huggingface_hub'], check=True)

print('✅ ComfyUI + H3/Director custom nodes installed.')
print('✅ No H3 model weights have been downloaded by this notebook yet.')


## 4. Load Cloudflare tunnel token + install cloudflared

Cloudflare Published Application must point to:

`comfy.zetbros.com` → `http://127.0.0.1:8188`

The Colab secret must be named `CF_TUNNEL_TOKEN`.

In [ ]:
import os, re, platform, subprocess
from google.colab import userdata

raw = userdata.get('CF_TUNNEL_TOKEN')
if not raw:
    raise RuntimeError('CF_TUNNEL_TOKEN is missing or notebook access is disabled.')

m = re.search(r'(eyJ[A-Za-z0-9._=-]+)', raw.strip())
if not m:
    raise RuntimeError('Could not find a Cloudflare Tunnel eyJ... token in CF_TUNNEL_TOKEN.')

CF_TUNNEL_TOKEN = m.group(1)
print('✅ Cloudflare tunnel token loaded from Colab Secrets. Token is not printed.')

arch = platform.machine().lower()
cf_arch = 'amd64' if arch in ('x86_64','amd64') else 'arm64' if arch in ('aarch64','arm64') else None
if not cf_arch:
    raise RuntimeError(f'Unsupported architecture: {arch}')

if subprocess.run(['bash','-lc','command -v cloudflared >/dev/null 2>&1']).returncode != 0:
    url = f'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{cf_arch}'
    subprocess.run(['curl','-fL','--retry','3','--retry-delay','2',url,'-o','/usr/local/bin/cloudflared'], check=True)
    subprocess.run(['chmod','0755','/usr/local/bin/cloudflared'], check=True)

print('✅', subprocess.check_output(['cloudflared','--version'], text=True).strip())


## 5. PRE-FLIGHT — Start ComfyUI + Cloudflare BEFORE model downloads

This is the checkpoint you asked for. When this cell succeeds, open `https://comfy.zetbros.com`. You should see the full ComfyUI interface, although model dropdowns will be empty/missing H3 weights until you continue later.

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

LOG_DIR = Path('/content/h3_comfy_logs')
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Clean only processes in this Colab runtime.
subprocess.run("pkill -f 'cf_origin_test_server.py'", shell=True, check=False)
subprocess.run("pkill -f 'cloudflared.*tunnel.*run'", shell=True, check=False)
subprocess.run("pkill -f 'python.*main.py.*--port 8188'", shell=True, check=False)
time.sleep(1)

comfy_log = open(LOG_DIR/'comfyui.log','w')
comfy_cmd = [
    sys.executable, 'main.py',
    '--listen', '127.0.0.1',
    '--port', '8188',
    '--disable-auto-launch',
    '--reserve-vram', H3_RESERVE_VRAM_GB,
    '--preview-method', 'none'
]
comfy_env = os.environ.copy()
comfy_env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
comfy_env['CUDA_MODULE_LOADING'] = 'LAZY'
comfy = subprocess.Popen(
    comfy_cmd, cwd='/content/ComfyUI',
    stdout=comfy_log, stderr=subprocess.STDOUT,
    env=comfy_env, start_new_session=True
)
(LOG_DIR/'comfyui.pid').write_text(str(comfy.pid))

for _ in range(120):
    time.sleep(1)
    if comfy.poll() is not None:
        raise RuntimeError('ComfyUI exited during startup. Check /content/h3_comfy_logs/comfyui.log')
    ok = subprocess.run(
        ['curl','-fsS','--connect-timeout','2','--max-time','3','http://127.0.0.1:8188/system_stats'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    ).returncode == 0
    if ok:
        break
else:
    raise RuntimeError('ComfyUI did not become healthy on 127.0.0.1:8188')

print('✅ ComfyUI local HTTP check: PASSED')

cf_log = open(LOG_DIR/'cloudflared.log','w')
cf = subprocess.Popen(
    ['cloudflared','tunnel','--no-autoupdate','run','--token',CF_TUNNEL_TOKEN],
    stdout=cf_log, stderr=subprocess.STDOUT, start_new_session=True
)
(LOG_DIR/'cloudflared.pid').write_text(str(cf.pid))

for _ in range(60):
    time.sleep(1)
    if cf.poll() is not None:
        raise RuntimeError('cloudflared exited during startup. Check /content/h3_comfy_logs/cloudflared.log')
    text = (LOG_DIR/'cloudflared.log').read_text(errors='replace') if (LOG_DIR/'cloudflared.log').exists() else ''
    if 'Registered tunnel connection' in text:
        break
else:
    raise RuntimeError('Cloudflare tunnel did not register within 60 seconds.')

print('✅ Cloudflare tunnel connection: REGISTERED')
print('🌐 OPEN NOW: https://comfy.zetbros.com')
print('⚠️ DO NOT CONTINUE TO MODEL DOWNLOADS until you personally confirm the full ComfyUI page opens.')


## 6. HARD STOP / approval gate

The default is `False`, so **Run all stops here before downloading any H3 weights**.

First open `https://comfy.zetbros.com`. If the full ComfyUI UI works, change the value below to `True` and run this cell again, then continue to Section 7.

In [ ]:
PROCEED_WITH_H3_MODEL_DOWNLOADS = False

if not PROCEED_WITH_H3_MODEL_DOWNLOADS:
    raise RuntimeError(
        'INTENTIONAL STOP: ComfyUI/Cloudflare preflight is complete. '
        'Verify https://comfy.zetbros.com first. Then set '
        'PROCEED_WITH_H3_MODEL_DOWNLOADS = True and rerun this cell.'
    )

print('✅ Approved. H3 model downloads may begin.')


## 7. Download the H3 model stack — only after approval

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import os, shutil

MODEL_ROOT = Path(f'{DRIVE_ROOT}/models' if PERSIST_MODELS_TO_DRIVE else '/content/ComfyUI/models')
for folder in ['diffusion_models','text_encoders','vae','loras','latent_upscale_models']:
    (MODEL_ROOT/folder).mkdir(parents=True, exist_ok=True)

if PERSIST_MODELS_TO_DRIVE:
    local_latent = Path('/content/ComfyUI/models/latent_upscale_models')
    drive_latent = MODEL_ROOT/'latent_upscale_models'
    drive_latent.mkdir(parents=True, exist_ok=True)
    if local_latent.is_symlink():
        local_latent.unlink()
    elif local_latent.exists():
        shutil.rmtree(local_latent) if local_latent.is_dir() else local_latent.unlink()
    local_latent.symlink_to(drive_latent, target_is_directory=True)

downloads = [
    ('Comfy-Org/MiniMax-H3','diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors', MODEL_ROOT, MODEL_ROOT/'diffusion_models'/'minimax_h3_fl2va_pruned_int8_convrot.safetensors'),
    ('Comfy-Org/MiniMax-H3','diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors', MODEL_ROOT, MODEL_ROOT/'diffusion_models'/'minimax_h3_ref2va_pruned_int8_convrot.safetensors'),
    ('Comfy-Org/MiniMax-H3','text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', MODEL_ROOT, MODEL_ROOT/'text_encoders'/'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'),
    ('Comfy-Org/MiniMax-H3','vae/minimax_h3_video_vae_fp16.safetensors', MODEL_ROOT, MODEL_ROOT/'vae'/'minimax_h3_video_vae_fp16.safetensors'),
    ('Comfy-Org/MiniMax-H3','vae/minimax_h3_audio_vae_fp32.safetensors', MODEL_ROOT, MODEL_ROOT/'vae'/'minimax_h3_audio_vae_fp32.safetensors'),
    ('lightx2v/Minimax-h3-Turbo','minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors', MODEL_ROOT/'loras', MODEL_ROOT/'loras'/'minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors'),
    ('lightx2v/Minimax-h3-Turbo','minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors', MODEL_ROOT/'loras', MODEL_ROOT/'loras'/'minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors'),
    ('LBH-123-AI/Minimax_h3_latent_Upscaler','minimax_h3_latent_upscaler_3d_fp16.safetensors', MODEL_ROOT/'latent_upscale_models', MODEL_ROOT/'latent_upscale_models'/'minimax_h3_latent_upscaler_3d_fp16.safetensors')
]

for repo_id, filename, local_dir, target in downloads:
    if target.exists() and target.stat().st_size > 1024*1024:
        print('SKIP', target.name)
        continue
    print('DOWNLOAD', filename)
    hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(local_dir))
    if not target.exists():
        raise FileNotFoundError(f'Expected model was not created: {target}')
    print('READY', target.name)

print('✅ H3 model stack ready at:', MODEL_ROOT)


## 8. Restart only ComfyUI so the new models appear

The Cloudflare connector is left running. The public URL stays the same.

In [ ]:
import subprocess, sys, time, os
from pathlib import Path

LOG_DIR = Path('/content/h3_comfy_logs')
subprocess.run("pkill -f 'python.*main.py.*--port 8188'", shell=True, check=False)
time.sleep(2)

comfy_log = open(LOG_DIR/'comfyui.log','w')
comfy_cmd = [
    sys.executable, 'main.py',
    '--listen', '127.0.0.1',
    '--port', '8188',
    '--disable-auto-launch',
    '--reserve-vram', H3_RESERVE_VRAM_GB,
    '--preview-method', 'none'
]
comfy_env = os.environ.copy()
comfy_env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
comfy_env['CUDA_MODULE_LOADING'] = 'LAZY'
comfy = subprocess.Popen(
    comfy_cmd, cwd='/content/ComfyUI',
    stdout=comfy_log, stderr=subprocess.STDOUT,
    env=comfy_env, start_new_session=True
)
(LOG_DIR/'comfyui.pid').write_text(str(comfy.pid))

for _ in range(120):
    time.sleep(1)
    if comfy.poll() is not None:
        raise RuntimeError('ComfyUI exited during final restart. Check comfyui.log')
    if subprocess.run(['curl','-fsS','--max-time','3','http://127.0.0.1:8188/system_stats'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0:
        break
else:
    raise RuntimeError('ComfyUI did not become healthy after model download.')

print('✅ FINAL READY')
print('🌐 https://comfy.zetbros.com')
print('Refresh the browser once; H3 model choices should now be available.')


## Diagnostics

Use this only if the preflight or final UI fails. It does not print the tunnel token.

In [ ]:
from pathlib import Path
import subprocess

LOG_DIR = Path('/content/h3_comfy_logs')
print('=== LOCAL COMFY ===')
subprocess.run("curl -sS -o /dev/null -w 'HTTP %{http_code}\n' http://127.0.0.1:8188/system_stats || true", shell=True)
subprocess.run("ss -ltnp | grep ':8188' || true", shell=True)
print('\n=== CLOUDFLARED ===')
subprocess.run("pgrep -af 'cloudflared.*tunnel.*run' | sed -E 's/(--token )[A-Za-z0-9._=-]+/\1[REDACTED]/g' || true", shell=True)
if (LOG_DIR/'cloudflared.log').exists():
    print('\n'.join((LOG_DIR/'cloudflared.log').read_text(errors='replace').splitlines()[-80:]))
print('\n=== COMFY LOG ===')
if (LOG_DIR/'comfyui.log').exists():
    print('\n'.join((LOG_DIR/'comfyui.log').read_text(errors='replace').splitlines()[-100:]))
